# Análise dos Resultados test5/test6 (Março 2026)

**Objetivo:** Analisar os 36 JSONs válidos da pasta `results/pre_phase__t5_and_t6_merged/` para:
1. Entender o estado atual dos dados antes de gastar GPU na Fase 2
2. Identificar se BoN > Pure já é estatisticamente claro com 3 seeds (ou se Fase 2 precisa de mais seeds)
3. Documentar o **confound de reward** entre BoN e Pure nos dados existentes
4. Suportar a decisão sobre qual(is) problema(s) excluir da Fase 2 (nguyen_1 = sem sinal)

**Status:** Parte da Fase 0.B do plano de dissertação (ver `docs/reports/THESIS_PLAN.md`)

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy import stats
from itertools import combinations

# Tenta importar statsmodels para Tukey HSD; graceful fallback se não disponível
try:
    from statsmodels.stats.multicomp import pairwise_tukeyhsd
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False
    print("statsmodels não encontrado. Instale com: pip install statsmodels")
    print("Pairwise t-tests serão usados como fallback.")

# Configurações de plot
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# Path para os dados (relativo a este notebook; ajuste se rodar de outro diretório)
DATA_DIR = Path('../../results/pre_phase__t5_and_t6_merged')
if not DATA_DIR.exists():
    # Fallback para path absoluto se o relativo não funcionar
    DATA_DIR = Path('results/pre_phase__t5_and_t6_merged')

print(f"Data dir: {DATA_DIR.resolve()}")
print(f"Exists: {DATA_DIR.exists()}")

## 1. Carregamento e Estruturação dos Dados

In [ ]:
def load_results(data_dir: Path) -> pd.DataFrame:
    """Carrega todos os JSONs e retorna DataFrame estruturado."""
    records = []
    for fpath in sorted(data_dir.glob('aggregate_*.json')):
        with open(fpath) as f:
            j = json.load(f)
        
        # Extrai campos do individual_results[0] para reward_fn e outros configs
        ind = j.get('individual_results', [{}])[0]
        
        records.append({
            'file': fpath.name,
            'algorithm': j.get('algorithm', 'unknown'),
            'problem': j.get('problem', 'unknown'),
            'seed': int(str(j.get('seeds', [0])[0])),
            'model': j.get('model', 'unknown').split('/')[-1],
            'mean_best_r2': j.get('mean_best_r2', np.nan),
            'mean_test_r2': j.get('mean_test_r2', np.nan),
            'std_best_r2': j.get('std_best_r2', np.nan),
            'reward_fn': ind.get('reward_fn', 'unknown'),
            'penalty_strategy': ind.get('penalty_strategy', 'unknown'),
            'temp_scheduler': ind.get('temp_scheduler', 'unknown'),
            'total_steps': ind.get('total_steps', np.nan),
            'total_unique_expressions': ind.get('total_unique_expressions', np.nan),
            'best_expression': ind.get('best_expression', ''),
        })
    
    df = pd.DataFrame(records)
    
    # Ordena problemas por dificuldade (definida por nível de complexidade da equação)
    problem_order = ['nguyen_1', 'nguyen_5', 'nguyen_9']
    algo_order = ['pure_ppo', 'pure_grpo', 'bon_ppo', 'bon_grpo']
    
    df['problem'] = pd.Categorical(df['problem'], categories=problem_order, ordered=True)
    df['algorithm'] = pd.Categorical(df['algorithm'], categories=algo_order, ordered=True)
    
    return df.sort_values(['algorithm', 'problem', 'seed'])


df = load_results(DATA_DIR)
print(f"Total de runs: {len(df)}")
print(f"\nDistribuição algorithm × problem:")
print(df.groupby(['algorithm', 'problem']).size().unstack(fill_value=0))
print(f"\nReward functions usadas por algoritmo:")
print(df.groupby(['algorithm', 'reward_fn']).size())

## 2. ⚠️ Confound Identificado: Reward Function

**Achado crítico:** Os algoritmos BoN usaram `sr_ic_lambda0.1` enquanto os algoritmos Pure usaram `r2_clipped`. 
Isso invalida qualquer comparação direta BoN vs. Pure nestes dados.

**Implicação para a Fase 2:** Todos os algoritmos devem usar o **mesmo reward** (`sr_ic`). Este confound é o principal motivo pelo qual a Fase 2 é necessária.

In [ ]:
print("=" * 60)
print("CONFOUND ANALYSIS: Reward Function por Algoritmo")
print("=" * 60)

confound_table = df.groupby(['algorithm', 'reward_fn'])['mean_best_r2'].agg(['count', 'mean', 'std'])
confound_table.columns = ['n_runs', 'mean_R2', 'std_R2']
confound_table['mean_R2'] = confound_table['mean_R2'].round(4)
confound_table['std_R2'] = confound_table['std_R2'].round(4)
print(confound_table)

print("\n⚠️  Pure PPO/GRPO = r2_clipped")
print("⚠️  BoN PPO/GRPO  = sr_ic_lambda0.1")
print("\nComparações BoN vs Pure INVÁLIDAS nestas dados (reward diferente).")
print("Fase 2 usa sr_ic uniforme para todos os algoritmos.")

## 3. Tabela Mestra: R² por Algoritmo × Problema × Seed

In [ ]:
print("=" * 80)
print("TABELA MESTRA: best_r2 por (algoritmo, problema, seed)")
print("=" * 80)

pivot = df.pivot_table(
    values='mean_best_r2',
    index='algorithm',
    columns=['problem', 'seed'],
    aggfunc='mean'
).round(4)

print(pivot.to_string())

print("\n" + "=" * 80)
print("RESUMO: mean ± std por (algoritmo, problema) — 3 seeds")
print("=" * 80)

summary = df.groupby(['algorithm', 'problem'])['mean_best_r2'].agg(['mean', 'std', 'min', 'max'])
summary.columns = ['mean_R2', 'std_R2', 'min_R2', 'max_R2']
summary = summary.round(4)
print(summary.to_string())

## 4. Análise Intra-Grupo (Comparações Válidas)

Dado o confound, comparações válidas são apenas:
- **Pure-PPO vs Pure-GRPO** (mesmo reward: `r2_clipped`)
- **BoN-PPO vs BoN-GRPO** (mesmo reward: `sr_ic`)
- Cada algoritmo vs. si mesmo entre problemas (análise de dificuldade)

In [ ]:
print("=" * 60)
print("COMPARAÇÃO VÁLIDA 1: Pure-PPO vs Pure-GRPO (r2_clipped)")
print("=" * 60)

pure = df[df['algorithm'].isin(['pure_ppo', 'pure_grpo'])]

for prob in ['nguyen_1', 'nguyen_5', 'nguyen_9']:
    ppo_vals = pure[(pure['algorithm'] == 'pure_ppo') & (pure['problem'] == prob)]['mean_best_r2'].values
    grpo_vals = pure[(pure['algorithm'] == 'pure_grpo') & (pure['problem'] == prob)]['mean_best_r2'].values
    
    if len(ppo_vals) >= 3 and len(grpo_vals) >= 3:
        t_stat, p_val = stats.ttest_ind(ppo_vals, grpo_vals)
        print(f"\n{prob}:")
        print(f"  Pure-PPO:  {ppo_vals.mean():.4f} ± {ppo_vals.std():.4f} (n={len(ppo_vals)})")
        print(f"  Pure-GRPO: {grpo_vals.mean():.4f} ± {grpo_vals.std():.4f} (n={len(grpo_vals)})")
        print(f"  t-test: t={t_stat:.3f}, p={p_val:.4f} {'*' if p_val < 0.05 else '(n.s.)'}")  
        print(f"  ⚠️  Aviso: 3 seeds é insuficiente para p<0.05 com variância alta.")

print("\n" + "=" * 60)
print("COMPARAÇÃO VÁLIDA 2: BoN-PPO vs BoN-GRPO (sr_ic)")
print("=" * 60)

bon = df[df['algorithm'].isin(['bon_ppo', 'bon_grpo'])]

for prob in ['nguyen_1', 'nguyen_5', 'nguyen_9']:
    ppo_vals = bon[(bon['algorithm'] == 'bon_ppo') & (bon['problem'] == prob)]['mean_best_r2'].values
    grpo_vals = bon[(bon['algorithm'] == 'bon_grpo') & (bon['problem'] == prob)]['mean_best_r2'].values
    
    if len(ppo_vals) >= 3 and len(grpo_vals) >= 3:
        t_stat, p_val = stats.ttest_ind(ppo_vals, grpo_vals)
        print(f"\n{prob}:")
        print(f"  BoN-PPO:  {ppo_vals.mean():.4f} ± {ppo_vals.std():.4f} (n={len(ppo_vals)})")
        print(f"  BoN-GRPO: {grpo_vals.mean():.4f} ± {grpo_vals.std():.4f} (n={len(grpo_vals)})")
        print(f"  t-test: t={t_stat:.3f}, p={p_val:.4f} {'*' if p_val < 0.05 else '(n.s.)'}")

## 5. Análise de Variância (ANOVA)

One-way ANOVA por problema (algoritmo como fator). Nota: com apenas 3 seeds o poder estatístico é baixo.

In [ ]:
print("=" * 60)
print("ANOVA por Problema (4 algoritmos × 3 seeds)")
print("=" * 60)
print("CAVEAT: Com 3 seeds/grupo, poder estatístico é baixo.")
print("Resultados são indicativos apenas. Fase 2 usa 5 seeds.\n")

for prob in ['nguyen_1', 'nguyen_5', 'nguyen_9']:
    groups = []
    labels = []
    for algo in ['pure_ppo', 'pure_grpo', 'bon_ppo', 'bon_grpo']:
        vals = df[(df['algorithm'] == algo) & (df['problem'] == prob)]['mean_best_r2'].values
        if len(vals) > 0:
            groups.append(vals)
            labels.append(algo)
    
    if len(groups) >= 2:
        f_stat, p_val = stats.f_oneway(*groups)
        print(f"{prob}: F={f_stat:.3f}, p={p_val:.4f} {'*** ANOVA significant p<0.05' if p_val < 0.05 else '(n.s.)'}")  
        for label, vals in zip(labels, groups):
            print(f"  {label:<12}: {vals.mean():.4f} ± {vals.std():.4f}")
        print()

print("\n=" * 61)
print("CONCLUSÃO PARA FASE 2:")
print("- nguyen_1: todos convergem (R²>0.99) — EXCLUIR da Fase 2")
print("- nguyen_5: alta variância, sem sinal claro com 3 seeds — INCLUIR com 5 seeds")
print("- nguyen_9: sinal moderado, variância alta — INCLUIR com 5 seeds")
print("- Adicionar nguyen_3 e nguyen_7 para cobertura de dificuldade")

## 6. Tukey HSD Post-Hoc (se statsmodels disponível)

In [ ]:
if HAS_STATSMODELS:
    print("=" * 60)
    print("TUKEY HSD — Comparações par-a-par por Problema")
    print("CAVEAT: Grupos homogêneos dentro do mesmo reward apenas")
    print("=" * 60)
    
    for prob in ['nguyen_5', 'nguyen_9']:  # Exclui nguyen_1 (sem variância)
        subset = df[df['problem'] == prob][['algorithm', 'mean_best_r2']].copy()
        subset['algorithm'] = subset['algorithm'].astype(str)
        
        if len(subset) >= 4:
            print(f"\n{prob}:")
            tukey = pairwise_tukeyhsd(
                endog=subset['mean_best_r2'],
                groups=subset['algorithm'],
                alpha=0.05
            )
            print(tukey)
else:
    print("statsmodels não disponível. Tukey HSD pulado.")
    print("Para instalação: pip install statsmodels")

## 7. Visualizações

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

problems = ['nguyen_1', 'nguyen_5', 'nguyen_9']
problem_titles = {
    'nguyen_1': 'Nguyen-1\n(easy: x + x² + x³)',
    'nguyen_5': 'Nguyen-5\n(medium: sin(x²)cos(x)-1)',
    'nguyen_9': 'Nguyen-9\n(hard, 2-var: sin(x₁)+sin(x₂²))'
}
colors = {'pure_ppo': '#2196F3', 'pure_grpo': '#03A9F4', 'bon_ppo': '#FF5722', 'bon_grpo': '#FF9800'}

for ax, prob in zip(axes, problems):
    data_prob = df[df['problem'] == prob]
    
    # Box plot por algoritmo
    algos = ['pure_ppo', 'pure_grpo', 'bon_ppo', 'bon_grpo']
    vals = [data_prob[data_prob['algorithm'] == a]['mean_best_r2'].values for a in algos]
    
    bp = ax.boxplot(vals, labels=algos, patch_artist=True, notch=False)
    
    for patch, algo in zip(bp['boxes'], algos):
        patch.set_facecolor(colors[algo])
        patch.set_alpha(0.7)
    
    # Scatter dos pontos individuais
    for i, (algo, v) in enumerate(zip(algos, vals), start=1):
        jitter = np.random.uniform(-0.1, 0.1, size=len(v))
        ax.scatter(np.ones(len(v)) * i + jitter, v, color=colors[algo], s=50, zorder=5, edgecolors='black', linewidth=0.5)
    
    ax.set_title(problem_titles.get(prob, prob), fontsize=10)
    ax.set_ylabel('Best R²' if ax == axes[0] else '')
    ax.set_xticklabels([a.replace('_', '-') for a in algos], rotation=30, ha='right', fontsize=8)
    ax.set_ylim(-0.1, 1.15)
    ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.axhline(y=0.9, color='green', linestyle=':', alpha=0.5, linewidth=0.8)

# Legenda de reward functions
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2196F3', alpha=0.7, label='Pure-PPO (r2_clipped)'),
    Patch(facecolor='#03A9F4', alpha=0.7, label='Pure-GRPO (r2_clipped)'),
    Patch(facecolor='#FF5722', alpha=0.7, label='BoN-PPO (sr_ic) ⚠️'),
    Patch(facecolor='#FF9800', alpha=0.7, label='BoN-GRPO (sr_ic) ⚠️'),
]
fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=4, fontsize=9)
fig.suptitle('Best R² por Algoritmo × Problema (test5/test6, n=3 seeds)\n⚠️ BoN usa reward diferente — comparação BoN vs Pure inválida', 
             fontsize=11, y=1.02)
plt.tight_layout()

out_dir = Path('../../docs/reports/figures')
out_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(out_dir / 't5_t6_boxplot.pdf', bbox_inches='tight')
plt.savefig(out_dir / 't5_t6_boxplot.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"Figura salva em {out_dir / 't5_t6_boxplot.pdf'}")

In [ ]:
# Heatmap: mean best_r2 por algoritmo × problema
fig, ax = plt.subplots(figsize=(8, 4))

heatmap_data = df.groupby(['algorithm', 'problem'])['mean_best_r2'].mean().unstack()
heatmap_data = heatmap_data[['nguyen_1', 'nguyen_5', 'nguyen_9']]  # ordem de dificuldade

sns.heatmap(
    heatmap_data, 
    annot=True, 
    fmt='.3f', 
    cmap='RdYlGn',
    vmin=0, vmax=1,
    ax=ax,
    linewidths=0.5,
    cbar_kws={'label': 'Mean Best R²'}
)

ax.set_title('Mean Best R² — test5/test6 (n=3 seeds/célula)\n⚠️ Confound reward: Pure=r2_clipped, BoN=sr_ic', fontsize=11)
ax.set_xlabel('Problema')
ax.set_ylabel('Algoritmo')
ax.set_xticklabels(['Nguyen-1\n(fácil)', 'Nguyen-5\n(médio)', 'Nguyen-9\n(difícil, 2-var)'])
ax.set_yticklabels([y.get_text().replace('_', '-') for y in ax.get_yticklabels()])

plt.tight_layout()
plt.savefig(out_dir / 't5_t6_heatmap.pdf', bbox_inches='tight')
plt.savefig(out_dir / 't5_t6_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"Heatmap salvo em {out_dir / 't5_t6_heatmap.pdf'}")

## 8. Análise das Melhores Expressões Encontradas

In [ ]:
print("=" * 80)
print("MELHORES EXPRESSÕES ENCONTRADAS (best_r2 >= 0.95)")
print("=" * 80)

top_runs = df[df['mean_best_r2'] >= 0.95].sort_values(['problem', 'mean_best_r2'], ascending=[True, False])

for _, row in top_runs.iterrows():
    print(f"\nProblema: {row['problem']} | Algoritmo: {row['algorithm']} | Seed: {row['seed']}")
    print(f"  best_r2={row['mean_best_r2']:.4f} | test_r2={row['mean_test_r2']:.4f}")
    print(f"  Expressão: {row['best_expression']}")
    print(f"  Reward: {row['reward_fn']} | Steps: {row['total_steps']}")

## 9. Decisão: O que os dados dizem sobre a Fase 2

In [ ]:
print("=" * 70)
print("DECISION SUMMARY — Implicações para Fase 2")
print("=" * 70)

# Analisa se nguyen_1 tem sinal
n1_vals = df[df['problem'] == 'nguyen_1']['mean_best_r2'].values
n5_vals = df[df['problem'] == 'nguyen_5']['mean_best_r2'].values
n9_vals = df[df['problem'] == 'nguyen_9']['mean_best_r2'].values

print(f"\nnguyen_1: min={n1_vals.min():.4f}, max={n1_vals.max():.4f}, std={n1_vals.std():.4f}")
print(f"nguyen_5: min={n5_vals.min():.4f}, max={n5_vals.max():.4f}, std={n5_vals.std():.4f}")
print(f"nguyen_9: min={n9_vals.min():.4f}, max={n9_vals.max():.4f}, std={n9_vals.std():.4f}")

print("\n" + "=" * 70)
print("DECISÕES:")
print()
print("1. EXCLUIR nguyen_1 da Fase 2:")
print(f"   Todos os algoritmos convergem (min R²={n1_vals.min():.4f}, std={n1_vals.std():.4f}).")
print("   Sem sinal discriminativo. Mantém como sanity check no piloto da Fase 1.")
print()
print("2. INCLUIR nguyen_5 e nguyen_9 com 5 seeds (não 3):")
print(f"   nguyen_5 std={n5_vals.std():.4f} — alta variância, 3 seeds insuficientes.")
print(f"   nguyen_9 std={n9_vals.std():.4f} — variância moderada, 3 seeds limítrofes.")
print("   5 seeds → power suficiente para Tukey HSD p<0.05.")
print()
print("3. CORRIGIR confound de reward antes da Fase 2:")
print("   Usar sr_ic para TODOS os algoritmos (best_of_n, pure_ppo, pure_grpo, bon_ppo, bon_grpo).")
print()
print("4. ADICIONAR nguyen_3 e nguyen_7 na Fase 2 para cobertura:")
print("   nguyen_3 = polinomial difícil, nguyen_7 = log-based (médio).")
print()
print("5. NÃO é necessário mudar o modelo base (Base-Infix):")
print("   Dados t5/t6 mostram que Base-Infix já tem capacidade de chegar a R²=1.0 (nguyen_9 pure_ppo seed42).")
print("   Confirmar na Fase 1 com piloto dos 6 modelos.")

## 10. Generalization Gap Analysis

Verifica se best_r2 (train) difere muito de test_r2 — indicativo de overfitting ao conjunto de validação.

In [ ]:
df['generalization_gap'] = df['mean_best_r2'] - df['mean_test_r2']

print("=" * 60)
print("GENERALIZATION GAP (best_r2 - test_r2)")
print("=" * 60)
print("Gap positivo grande = expressão ajusta bem ao ponto best mas não generaliza")
print()

gap_summary = df.groupby(['algorithm', 'problem'])['generalization_gap'].agg(['mean', 'std', 'max'])
gap_summary.columns = ['mean_gap', 'std_gap', 'max_gap']
gap_summary = gap_summary.round(4)
print(gap_summary.to_string())

# Identifica casos com gap extremo (> 100, por problemas numéricos)
extreme = df[df['generalization_gap'].abs() > 10]
if len(extreme) > 0:
    print(f"\n⚠️  {len(extreme)} runs com gap extremo (|gap| > 10):")
    print(extreme[['algorithm', 'problem', 'seed', 'mean_best_r2', 'mean_test_r2', 'best_expression']].to_string())
    print("\nEstes casos indicam expressões que colapso numérico (divisão por zero, overflow).")
    print("A Fase 2 deve filtrar test_r2 < -1000 como 'failed run' antes da análise estatística.")

## 11. Resumo Final

In [ ]:
print("=" * 70)
print("RESUMO FINAL — Análise t5/t6")
print("=" * 70)
print(f"""
Dados analisados: {len(df)} runs
Design: 4 algoritmos × 3 problemas × 3 seeds = 36 runs
Modelo: gpt2_base_infix_682k (Base, 124M, Infix)
Fonte: results/pre_phase__t5_and_t6_merged/

ACHADOS PRINCIPAIS:

1. CONFOUND (crítico): BoN usa sr_ic_lambda0.1; Pure usa r2_clipped.
   → Comparação BoN vs Pure inválida nestes dados.
   → Fase 2 deve usar reward uniforme para todos.

2. nguyen_1 é trivial (R²≥0.99 para todos) → excluir da Fase 2.

3. nguyen_5 e nguyen_9 mostram variância alta com 3 seeds → 5 seeds na Fase 2.

4. Expressões com gap extremo (test_r2 << 0) existem em alguns runs.
   → Adicionar filtro test_r2 < -1000 como critério de 'failed run'.

5. Base-Infix já consegue R²=1.0 em nguyen_9 (pure_ppo, seed42).
   → Evidência de capacidade; Fase 1 confirma com piloto comparativo.

PRÓXIMOS PASSOS (Fase 1):
- Corrigir best_of_n.py:254 (RewardResult wrapper)
- Corrigir run_experiment.py:163 (bug 2-variáveis)
- Piloto zero-shot nos 6 modelos para selecionar o melhor
- Scout runs de plateau (determine MAX_STEPS para Fase 2)
""")